##CScorpus
CSCorpus is a curated computer science corpus consisting of technical Wikipedia articles, StackExchange discussions, software documentation, and educational notes. It was built to train a domain-specific GPT model with strong coverage of algorithms, mathematics, programming, systems, and machine learning concepts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/GPT_from_scratch"

folders = [
    "data/raw",
    "data/processed",
    "checkpoints",
    "logs",
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Folders created")

In [ ]:
!pip install -U datasets huggingface_hub pyarrow

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1"
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/GPT_from_scratch"

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/data", exist_ok=True)
os.makedirs(f"{BASE_DIR}/data/raw", exist_ok=True)
os.makedirs(f"{BASE_DIR}/data/processed", exist_ok=True)

print("Folders created")

In [ ]:
all_text = "\n".join(dataset["train"]["text"])

path = f"{BASE_DIR}/data/raw/wikitext103.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write(all_text)

print("saved")

In [ ]:
topics = [
    # Foundations & Math
    "Linear algebra", "Probability theory", "Statistics", "Calculus for machine learning", "Optimization theory", "Information theory", "Discrete mathematics",

    # Systems & Hardware
    "Operating system", "Computer architecture", "Memory management", "Distributed systems", "Cloud computing", "Containerization (Docker)", "Parallel computing", "Compiler design",

    # Data Structures & Algorithms
    "Data structure", "Algorithm", "Dynamic programming", "Graph theory", "Sorting and searching", "Complexity analysis (Big O)", "Greedy algorithms", "Hash tables",

    # Databases & Networking
    "Database management systems", "SQL", "NoSQL databases", "Computer network", "TCP/IP protocol suite", "REST API", "Database indexing",

    # Machine Learning & Deep Learning
    "Artificial intelligence", "Machine learning", "Deep learning", "Neural network", "Transformer (deep learning architecture)", "Large language model", "Reinforcement learning", "Transfer learning", "Backpropagation", "Gradient descent", "Regularization", "Attention mechanism",

    # Applied ML & MLOps
    "MLOps", "Data versioning", "Experiment tracking", "Model deployment", "Retrieval-augmented generation (RAG)", "Vector databases", "Tokenization", "Hyperparameter tuning",

    # Programming & Frameworks
    "Python programming language", "NumPy", "PyTorch", "Scikit-learn", "Pandas", "C++", "System design", "Object-oriented programming", "Unit testing", "Git"
]

In [ ]:
!pip install wikipedia -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/GPT_from_scratch"

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/data", exist_ok=True)
os.makedirs(f"{BASE_DIR}/data/raw", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)

OUTPUT_FILE = f"{BASE_DIR}/data/raw/cs_corpus_recursive_wiki.jsonl"

print("Folders Ready")
print("Saving to:")
print(OUTPUT_FILE)

In [ ]:
!pip install wikipedia wikipedia-api -q

In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    user_agent="CSGPT_Crawler/1.0 (Akash Kumar Thakur; NIE Mysuru)",
    language="en"
)

In [ ]:
SEEDS = [
    "Machine learning", "Deep learning", "Artificial intelligence", "Neural network",
    "Transformer (deep learning architecture)", "Large language model", "Data structure",
    "Algorithm", "Dynamic programming", "Graph theory", "Operating system", "Database",
    "Computer network", "Python (programming language)", "NumPy", "PyTorch",
    "Scikit-learn", "Linear algebra", "Probability theory", "Statistics",
    "Calculus", "Optimization theory", "Information theory", "Discrete mathematics",
    "Computer architecture", "Memory management", "Distributed computing",
    "Cloud computing", "Docker (software)", "Parallel computing", "Compiler design",
    "Sorting algorithm", "Big O notation", "Greedy algorithm", "Hash table",
    "Database management system", "SQL", "NoSQL", "TCP/IP", "REST API",
    "Backpropagation", "Gradient descent", "Regularization (mathematics)",
    "Attention (machine learning)", "MLOps", "Data versioning",
    "Retrieval-augmented generation", "Vector database", "Tokenization",
    "Hyperparameter optimization", "Pandas (software)", "C++", "System design",
    "Object-oriented programming", "Unit testing", "Git"
]
CS_KEYWORDS = {

    "algorithm",
    "data",
    "computer",
    "network",
    "database",
    "machine",
    "learning",
    "neural",
    "transformer",
    "artificial",
    "intelligence",
    "programming",
    "python",
    "software",
    "compiler",
    "distributed",
    "statistics",
    "probability",
    "mathematics",
    "optimization",
    "pytorch",
    "numpy",
    "pandas",
    "scikit"
}


In [ ]:
import wikipediaapi
import time
import json
from collections import deque
from tqdm import tqdm
import time

MAX_PAGES = 2000
MAX_DEPTH = 3

visited = set()

queue = deque()

def is_relevant(title):
    t = title.lower()
    return any(k in t for k in CS_KEYWORDS)

def get_page_with_retry(title, retries=3):
    for i in range(retries):
        try:
            page = wiki.page(title)
            if page.exists():
                return page
        except:
            time.sleep(2 ** i)
    return None

def crawl():
    visited = set()
    queue = deque([(seed, 0) for seed in SEEDS])
    pages_saved = 0

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

        while queue and pages_saved < MAX_PAGES:

            title, depth = queue.popleft()

            if title in visited:
                continue

            page = get_page_with_retry(title)

            if page is None:
                continue

            if len(page.text) < 3000:
                continue

            visited.add(title)

            data = {
                "title": title,
                "depth": depth,
                "text": page.text
            }

            f.write(json.dumps(data, ensure_ascii=False) + "\n")

            pages_saved += 1

            print(f"[{pages_saved}] {title}")

            if depth < MAX_DEPTH:

                for link in page.links.keys():

                    if link not in visited and is_relevant(link):
                        queue.append((link, depth + 1))

            time.sleep(0.25)

    stats = {
        "pages_saved": pages_saved,
        "visited": len(visited)
    }

    with open(f"{BASE_DIR}/logs/wiki_stats.json", "w") as fp:
        json.dump(stats, fp, indent=4)

    return stats

stats = crawl()

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import json
import tiktoken

dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="default",
    split="train",
    streaming=True
)

OUTPUT_FILE2 = f"{BASE_DIR}/data/raw/HF_corpus.jsonl"

enc = tiktoken.get_encoding("gpt2")

TARGET_TOKENS = 250_000_000

def stream_to_disk():

    total_tokens = 0
    total_docs = 0

    with open(OUTPUT_FILE2, "w", encoding="utf-8") as f:

        for sample in tqdm(dataset):

            text = sample["text"]

            if len(text) < 500:
                continue

            token_count = len(enc.encode(text))

            f.write(
                json.dumps(
                    {"text": text},
                    ensure_ascii=False
                ) + "\n"
            )

            total_tokens += token_count
            total_docs += 1

            if total_tokens >= TARGET_TOKENS:
                break

    print(f"Docs Saved   : {total_docs:,}")
    print(f"Tokens Saved : {total_tokens:,}")

stream_to_disk()

In [ ]:
import os

size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024

print(f"Corpus Size: {size_mb:.2f} MB")

In [ ]:
import json
import tiktoken

enc = tiktoken.get_encoding("gpt2")

token_count = 0

with open(OUTPUT_FILE, encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        token_count += len(enc.encode(data["text"]))

print(f"Tokens: {token_count:,}")

###BINNING

### Binning (train.bin / val.bin)

**Definition**

Binning is the preprocessing step where all text documents are tokenized once using the GPT tokenizer and stored as a contiguous binary file (`train.bin` and `val.bin`) instead of keeping them as raw text.

---

### Why Binning?

Without binning:

```text
Raw Text
   ↓
Tokenize
   ↓
Train Epoch 1

Raw Text
   ↓
Tokenize Again
   ↓
Train Epoch 2
```

The tokenizer runs every epoch, wasting CPU time and slowing training.

With binning:

```text
Raw Text
   ↓
Tokenize Once
   ↓
train.bin / val.bin
   ↓
Training
```

Tokenization is performed only once, making training significantly faster.

---

### Why Binary Files?

GPT-2 vocabulary size:

```text
50,257 tokens
```

Each token can be stored as:

```python
uint16
```

which requires only:

```text
2 bytes per token
```

Example:

```text
500M tokens
≈ 1 GB storage
```

making binary storage highly space-efficient.

---

### Advantages

* Tokenization happens only once.
* Faster training startup.
* Lower CPU overhead during training.
* Minimal RAM usage through memory mapping (`np.memmap`).
* Scales to hundreds of millions of tokens.
* Same approach used by GPT-style implementations such as nanoGPT.

---

### Final Pipeline

```text
CSCorpus + FineWeb-Edu + WikiText103
                ↓
           Tokenization
                ↓
      train.bin + val.bin
                ↓
          Memory Mapping
                ↓
            DataLoader
                ↓
            GPT Model
```

---

### Memory Mapping

Instead of loading all tokens into RAM:

```python
train_data = np.memmap(
    "train.bin",
    dtype=np.uint16,
    mode="r"
)
```

only the required token chunk is read during training:

```python
x = train_data[idx : idx + context_length]
```

This allows training on very large corpora without requiring large system memory.

---

**In one sentence:**

> Binning converts the entire corpus into compact token IDs stored in binary files so that GPT training can stream tokens directly from disk efficiently without repeatedly tokenizing raw text.


In [ ]:
import json
import numpy as np
import array, random
import tiktoken
from tqdm import tqdm

In [ ]:
CORPUS_FILES = [
    "/content/drive/MyDrive/GPT_from_scratch/data/raw/cs_corpus_recursive_wiki.jsonl",
    "/content/drive/MyDrive/GPT_from_scratch/data/raw/HF_corpus.jsonl",
    "/content/drive/MyDrive/GPT_from_scratch/data/raw/wikitext103.txt"
]

enc = tiktoken.get_encoding("gpt2")

TRAIN_BIN = "/content/drive/MyDrive/GPT_from_scratch/data/processed/train.bin"
VAL_BIN = "/content/drive/MyDrive/GPT_from_scratch/data/processed/val.bin"

In [ ]:
TRAIN_RATIO = 0.9
BUFFER_SIZE = 5_000_000

train_buffer = array.array("H")
val_buffer = array.array("H")

train_tokens = 0
val_tokens = 0

train_f = open(TRAIN_BIN, "wb")
val_f = open(VAL_BIN, "wb")

for file in CORPUS_FILES:

    print("Processing:", file)

    if file.endswith(".jsonl"):

        with open(file, encoding="utf-8") as f:

            for line in tqdm(f):

                try:
                    text = json.loads(line)["text"]
                except:
                    continue

                ids = enc.encode_ordinary(text)

                if random.random() < TRAIN_RATIO:

                    train_buffer.extend(ids)
                    train_tokens += len(ids)

                    if len(train_buffer) >= BUFFER_SIZE:
                        train_buffer.tofile(train_f)
                        train_buffer = array.array("H")

                else:

                    val_buffer.extend(ids)
                    val_tokens += len(ids)

                    if len(val_buffer) >= BUFFER_SIZE:
                        val_buffer.tofile(val_f)
                        val_buffer = array.array("H")

    else:

        with open(file, encoding="utf-8") as f:
            text = f.read()

        ids = enc.encode_ordinary(text)

        split = int(len(ids) * TRAIN_RATIO)

        train_buffer.extend(ids[:split])
        val_buffer.extend(ids[split:])

        train_tokens += split
        val_tokens += len(ids) - split

if len(train_buffer):
    train_buffer.tofile(train_f)

if len(val_buffer):
    val_buffer.tofile(val_f)

train_f.close()
val_f.close()

print(f"Train Tokens: {train_tokens:,}")
print(f"Val Tokens: {val_tokens:,}")

In [ ]:
import os

size_gb = os.path.getsize(TRAIN_BIN) / 1024**3

print(f"Size: {size_gb:.2f} GB")

In [ ]:
import numpy as np

tokens = np.memmap(
    TRAIN_BIN,
    dtype=np.uint16,
    mode="r"
)

print(f"Tokens: {len(tokens):,}")

<img src="https://camo.githubusercontent.com/5435c19ef837e8a269b9f829af032ce0103519a6bd3d1030abdda58a6db498a1/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31372e77656270" width=700>

In [ ]:
import gc
import torch

# Clear active garbage collector blocks and flush PyTorch memory allocations
gc.collect()
torch.cuda.empty_cache()

###ARCHITECTURE

In [ ]:
import os
import json
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#MODEL 1
"""VOCAB_SIZE = 50257
CONTEXT_LENGTH = 256

D_MODEL = 512
N_HEADS = 8
N_LAYERS = 8

DROPOUT = 0.1
CHECKPOINT_DIR = "/content/drive/MyDrive/GPT_from_scratch/checkpoints"

"""
#model 1 had like 4.4 loss at 30K steps
# =====================================================================
# ITERATION 2: SCALED HYPERPARAMETERS (SAME ARCHITECTURE)
# =====================================================================
VOCAB_SIZE = 50257
CONTEXT_LENGTH = 256

D_MODEL = 512              # Kept fixed to prevent VRAM allocation crashes
N_HEADS = 8                # Kept fixed so head_dim stays exactly 64 (512 // 8)
N_LAYERS = 12              # UPGRADED: Increased depth from 8 to 12 blocks

DROPOUT = 0.0              #  Drop to 0.0, use Weight Decay in AdamW instead
CHECKPOINT_DIR = "/content/drive/MyDrive/GPT_from_scratch/checkpoints_model2"


print(DEVICE)

TOKENIZER(not needed as we have binned the tokenized data for efficient retrival)

In [ ]:
!pip install tiktoken -q

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

##Data Loader

In [ ]:
import numpy as np

TRAIN_BIN = "/content/drive/MyDrive/GPT_from_scratch/data/processed/train.bin"
VAL_BIN = "/content/drive/MyDrive/GPT_from_scratch/data/processed/val.bin"

train_data = np.memmap(TRAIN_BIN,dtype=np.uint16,mode="r")
val_data = np.memmap(VAL_BIN,dtype=np.uint16,mode="r")

print(len(train_data))
print(len(val_data))

In [ ]:
def get_batch(split,batch_size):

    data = train_data if split=="train" else val_data

    idx = torch.randint(
        len(data)-CONTEXT_LENGTH-1,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            data[i:i+CONTEXT_LENGTH].astype(np.int64)
        )
        for i in idx
    ])

    y = torch.stack([
        torch.from_numpy(
            data[i+1:i+CONTEXT_LENGTH+1].astype(np.int64)
        )
        for i in idx
    ])

    return x.to(DEVICE), y.to(DEVICE)

##Embedding layer

In [ ]:
class GPTEmbedding(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LENGTH,
            D_MODEL
        )

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self,x):

        B,T = x.shape

        positions = torch.arange(
            T,
            device=x.device
        )

        tok = self.token_embedding(x)
        pos = self.position_embedding(positions)

        return self.dropout(tok + pos)

##Attention block

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self):
        super().__init__()

        assert D_MODEL % N_HEADS == 0

        self.qkv = nn.Linear(
            D_MODEL,
            3 * D_MODEL
        )

        self.proj = nn.Linear(
            D_MODEL,
            D_MODEL
        )

        self.resid_dropout = nn.Dropout(
            DROPOUT
        )

    def forward(self, x):

        B, T, C = x.shape

        head_dim = C // N_HEADS

        qkv = self.qkv(x)

        q, k, v = qkv.chunk(
            3,
            dim=-1
        )

        q = q.view(
            B, T,
            N_HEADS,
            head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            N_HEADS,
            head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            N_HEADS,
            head_dim
        ).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=DROPOUT if self.training else 0.0,
            is_causal=True
        )

        out = (
            out.transpose(1, 2)
            .contiguous()
            .view(B, T, C)
        )

        return self.resid_dropout(
            self.proj(out)
        )

##FFN

In [ ]:
class FeedForward(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                D_MODEL,
                4*D_MODEL
            ),
            nn.GELU(),
            nn.Linear(
                4*D_MODEL,
                D_MODEL
            ),
            nn.Dropout(DROPOUT)
        )

    def forward(self,x):
        return self.net(x)

##Transformer

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(D_MODEL)
        self.ln2 = nn.LayerNorm(D_MODEL)

        self.attn = MultiHeadAttention()
        self.ffn = FeedForward()

    def forward(self,x):

        x = x + self.attn(
            self.ln1(x)
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

##GPT

In [ ]:
class GPTModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = GPTEmbedding()

        self.blocks = nn.ModuleList([
            TransformerBlock()
            for _ in range(N_LAYERS)
        ])

        self.ln_f = nn.LayerNorm(
            D_MODEL
        )

        self.lm_head = nn.Linear(
            D_MODEL,
            VOCAB_SIZE,
            bias=False
        )

        self.lm_head.weight = (
            self.embedding
            .token_embedding
            .weight
        )

        self.apply(self.init_weights)

    def init_weights(self,module):

      if isinstance(module, nn.Linear):

          torch.nn.init.normal_(
              module.weight,
              mean=0.0,
              std=0.02
          )

          if module.bias is not None:

              torch.nn.init.zeros_(
                  module.bias
              )

      elif isinstance(module, nn.Embedding):

          torch.nn.init.normal_(
              module.weight,
              mean=0.0,
              std=0.02
          )

    def forward(
        self,
        idx,
        targets=None
    ):

        x = self.embedding(idx)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                targets.reshape(-1)
            )

        return logits, loss

In [ ]:
model = GPTModel().to(DEVICE)

print(
    sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6,
    "Million Params"
)

In [ ]:
#sanity check
x,y = get_batch(
    "train",
    batch_size=4
)

logits,loss = model(x,y)

print(logits.shape)
print(loss.item())

In [ ]:
import os

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

def save_checkpoint(
    model,
    optimizer,
    step,
    loss
):

    path = (
        f"{CHECKPOINT_DIR}/"
        f"step_{step}.pt"
    )

    torch.save(
        {
            "step": step,
            "loss": loss,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict()
        },
        path
    )

    print("Saved:", path)

In [ ]:
print(D_MODEL)
print(N_HEADS)
print(N_LAYERS)

In [ ]:
print(
    model.lm_head.weight.data_ptr()
)

print(
    model.embedding.token_embedding.weight.data_ptr()
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9,0.95),
    weight_decay=0.1
)

In [ ]:
decay = []
no_decay = []

for name,param in model.named_parameters():

    if param.dim() >= 2:
        decay.append(param)

    else:
        no_decay.append(param)

optimizer = torch.optim.AdamW(
    [
        {
            "params": decay,
            "weight_decay": 0.1
        },
        {
            "params": no_decay,
            "weight_decay": 0.0
        }
    ],
    lr=3e-4,
    betas=(0.9,0.95)
)

In [ ]:
#LR sheduler
MAX_STEPS = 50000
WARMUP_STEPS = 1000

def get_lr(step):

    if step < WARMUP_STEPS:

        return 3e-4 * step / WARMUP_STEPS

    progress = (
        step - WARMUP_STEPS
    ) / (
        MAX_STEPS - WARMUP_STEPS
    )

    return (
        0.1 * 3e-4
        +
        0.5 * (3e-4 - 0.1*3e-4)
        * (1 + math.cos(math.pi * progress))
    )

In [ ]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    losses = {}

    for split in ["train","val"]:

        vals = []

        for _ in range(20):

            x,y = get_batch(
                split,
                batch_size=32
            )

            _,loss = model(x,y)

            vals.append(
                loss.item()
            )

        losses[split] = sum(vals)/len(vals)

    model.train()

    return losses

In [ ]:
os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

def save_checkpoint(
    step,
    train_loss,
    val_loss
):

    path = (
        f"{CHECKPOINT_DIR}/"
        f"checkpoint_{step}.pt"
    )

    torch.save(
        {
            "step": step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss
        },
        path
    )

    print("Saved:",path)

def load_checkpoint(path):

    ckpt = torch.load(
        path,
        map_location=DEVICE
    )

    model.load_state_dict(
        ckpt["model"]
    )

    optimizer.load_state_dict(
        ckpt["optimizer"]
    )

    print(
        "Loaded step",
        ckpt["step"]
    )

    return ckpt["step"]

In [ ]:
@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=100,
    temperature=1.0,
    top_k=50
):

    model.eval()

    ids = tokenizer.encode(prompt)

    x = torch.tensor(
        ids,
        dtype=torch.long,
        device=DEVICE
    )[None,:]

    for _ in range(max_new_tokens):

        x_cond = x[:,-CONTEXT_LENGTH:]

        logits,_ = model(x_cond)

        logits = logits[:,-1,:]

        logits /= temperature

        v,_ = torch.topk(
            logits,
            top_k
        )

        logits[
            logits < v[:,-1,None]
        ] = -float("inf")

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            1
        )

        x = torch.cat(
            [x,next_token],
            dim=1
        )

    return tokenizer.decode(
        x[0].tolist()
    )

In [ ]:
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE=="cuda")
)

In [ ]:
#train

EVAL_INTERVAL = 100
SAVE_INTERVAL = 100
MAX_STEPS = 50000

for step in range(MAX_STEPS):

    lr = get_lr(step)

    for pg in optimizer.param_groups:
        pg["lr"] = lr

    x, y = get_batch(
        "train",
        batch_size=32
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE,
        dtype=torch.float16,
        enabled=(DEVICE == "cuda")
    ):

        _, loss = model(x, y)

    if torch.isnan(loss):

        print(
            f"NaN detected at step {step}"
        )

        break

    scaler.scale(loss).backward()

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    scaler.step(optimizer)

    scaler.update()

    if step % 10 == 0:

        print(
            f"step={step} "
            f"loss={loss.item():.4f} "
            f"lr={lr:.6e}"
        )

    if step % 25 == 0:

        save_checkpoint(
            step,
            loss.item(),
            -1
        )

print("Training Complete")

In [ ]:
generate(
    "Machine learning is",
    max_new_tokens=100,
    temperature=0.8
)

"""
MODEL1 output
Machine learning is being also about the report.
Tyli, and have been more than many children.
- If some of the study are very much of the problem.
- You are often in the time.
- What would also be a few years who didn't see it. We will be the one more than the state that the first had an important time that these people were more than 100.
- 2012, the city may be a large process of it to provide the age of the
"""

In [ ]:
import os
import re
import math
import torch

CHECKPOINT_DIR = "/content/drive/MyDrive/GPT_from_scratch/checkpoints"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

def save_checkpoint(
    step,
    train_loss,
    val_loss
):

    path = (
        f"{CHECKPOINT_DIR}/"
        f"checkpoint_{step}.pt"
    )

    torch.save(
        {
            "step": step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss
        },
        path
    )

    print(
        f"Saved: {path}"
    )

def get_latest_checkpoint():

    files = [
        f
        for f in os.listdir(
            CHECKPOINT_DIR
        )
        if f.endswith(".pt")
    ]

    if len(files) == 0:
        return None

    files.sort(
        key=lambda x:
        int(
            re.findall(
                r"\d+",
                x
            )[0]
        )
    )

    return os.path.join(
        CHECKPOINT_DIR,
        files[-1]
    )

def load_checkpoint(path):

    checkpoint = torch.load(
        path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    start_step = (
        checkpoint["step"] + 1
    )

    print(
        f"Resumed from step "
        f"{checkpoint['step']}"
    )

    return start_step

@torch.no_grad()
def estimate_loss(
    eval_iters=100,
    batch_size=16
):

    model.eval()

    metrics = {}

    for split in [
        "train",
        "val"
    ]:

        losses = []

        for _ in range(
            eval_iters
        ):

            x, y = get_batch(
                split,
                batch_size
            )

            _, loss = model(
                x,
                y
            )

            losses.append(
                loss.item()
            )

        avg_loss = (
            sum(losses)
            /
            len(losses)
        )

        metrics[split] = {
            "loss": avg_loss,
            "ppl": math.exp(
                avg_loss
            )
        }

    model.train()

    print(
        "\n"
        + "=" * 60
    )

    print(
        f"Train Loss : "
        f"{metrics['train']['loss']:.4f}"
    )

    print(
        f"Train PPL  : "
        f"{metrics['train']['ppl']:.2f}"
    )

    print()

    print(
        f"Val Loss   : "
        f"{metrics['val']['loss']:.4f}"
    )

    print(
        f"Val PPL    : "
        f"{metrics['val']['ppl']:.2f}"
    )

    print(
        "=" * 60
    )

    return metrics

latest_checkpoint = (
    get_latest_checkpoint()
)

if latest_checkpoint is not None:

    START_STEP = load_checkpoint(
        latest_checkpoint
    )

else:

    START_STEP = 0

    print(
        "No checkpoint found."
        " Starting fresh."
    )

MAX_STEPS = 50000

EVAL_INTERVAL = 500

SAVE_INTERVAL = 500

for step in range(
    START_STEP,
    MAX_STEPS
):

    lr = get_lr(
        step
    )

    for pg in optimizer.param_groups:

        pg["lr"] = lr

    x, y = get_batch(
        "train",
        batch_size=32
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE,
        dtype=torch.float16,
        enabled=(
            DEVICE == "cuda"
        )
    ):

        _, loss = model(
            x,
            y
        )

    if torch.isnan(
        loss
    ):

        print(
            f"NaN detected "
            f"at step {step}"
        )

        break

    scaler.scale(
        loss
    ).backward()

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    if step % 100 == 0:

        print(
            f"step={step} "
            f"loss={loss.item():.4f} "
            f"lr={lr:.6e}"
        )

    if step % EVAL_INTERVAL == 0:

        metrics = estimate_loss(
            eval_iters=100
        )

    if step % SAVE_INTERVAL == 0:

        save_checkpoint(
            step,
            metrics["train"]["loss"],
            metrics["val"]["loss"]
        )

print(
    "Training Complete"
)

##EVALUATION

In [ ]:
@torch.no_grad()
def estimate_loss2(
    eval_iters=200,
    batch_size=32
):

    model.eval()

    metrics = {}

    for split in ["train", "val"]:

        losses = []

        for _ in range(eval_iters):

            x, y = get_batch(
                split,
                batch_size
            )

            _, loss = model(x, y)

            losses.append(
                loss.item()
            )

        avg_loss = sum(losses) / len(losses)

        metrics[split] = {
            "loss": avg_loss,
            "ppl": math.exp(avg_loss)
        }

    model.train()

    print("\n" + "=" * 60)

    print(f"Train Loss : {metrics['train']['loss']:.4f}")

    print(f"Train perplexity  : {metrics['train']['ppl']:.2f}")

    print()

    print(f"Val Loss   : {metrics['val']['loss']:.4f}")

    print(f"Val perplexity    : {metrics['val']['ppl']:.2f}")

    print("=" * 60)

    return metrics

metrics = estimate_loss2(eval_iters=200)

###EVAL GENNERATION

In [ ]:
import os
import json
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

#a eval on CPU
CHECKPOINT_PATH= "/content/drive/MyDrive/GPT_from_scratch/checkpoints/checkpoint_31625.pt"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

model.load_state_dict(
    checkpoint["model"]
)

model.eval()

print(
    f"Loaded step {checkpoint['step']}"
)

In [ ]:
@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    top_k=50
):

    ids = tokenizer.encode(
        prompt
    )

    x = torch.tensor(
        ids,
        dtype=torch.long
    ).unsqueeze(0)

    for _ in range(
        max_new_tokens
    ):

        x_cond = x[
            :,
            -CONTEXT_LENGTH:
        ]

        logits, _ = model(
            x_cond
        )

        logits = logits[:,-1,:]

        logits = (logits/ temperature        )

        v, _ = torch.topk(
            logits,
            top_k
        )

        logits[
            logits
            <
            v[:, -1, None]
        ] = -float("inf")

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = (
            torch.multinomial(
                probs,
                1
            )
        )

        x = torch.cat([x,next_token],dim=1)

    return tokenizer.decode(
        x[0].tolist()
    )

In [ ]:
!pip install tiktoken -q

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
prompts = [
    "Machine learning is",
    "Neural networks are",
    "The transformer architecture",
    "Gradient descent works by",
    "Python programming",
    "Artificial intelligence is"
]

for p in prompts:

    print("=" * 80)

    print(
        generate(
            p,
            max_new_tokens=150
        )
    )

    print()

"""
================================================================================
Machine learning is not merely the same as the human consciousness.
“I think we’re being taught to be taught to be in a language that they’re no longer required to study.”
“The cognitive abilities of the human mind are important to the development of consciousness.
The human mind’s ability to adapt to the physical world is critical for understanding that is essential to the development of the cognitive abilities of the mind. It is a great opportunity for the individual to develop and maintain the capacity to understand the needs of their individual and group.
“At the beginning of the world the mind’s role in the creation of the mind is the necessary basis for understanding its own identity and to create the identity

================================================================================
Neural networks are the central regions of the human body that are important for the functioning of the body.
An important consequence of this process is that the brain function is more important than the function of organs and organs, which are vital for the functioning of the body. These processes are not directly involved in the normal functioning of the body and are part of the system.
In the biological process, there are many components of the system, which include the body, body, and body.
A system is the basis of the physical system and is the system of the system.
This means that the system can be connected to the body.
A system is the system of a system which is vital for the body.
A system is an organ of the body

================================================================================
The transformer architecture is the most useful way to deal with the design of the interface.

KOSA's architecture of a computer is called a "V-based" machine. When it comes to an operating system, it's not always possible to run a computer on a PC. The processor is designed to deal with the problem of performing a system interface.
In some cases it's hard to run the computer when the computer is running a computer.
In some cases the PC can do a variety of things, such as:

Liver (a type of computer that contains a device called a computer that is a computer that is connected to a computer) and a physical computer that contains the computer.
In some cases it's hard to run

================================================================================
Gradient descent works by using a pre-existing system, the method can also be used as a starting point for the development of a new program.
As a result, the program has many of the advantages and disadvantages of a system. This is because the program can be used to produce and develop more efficient program.
In the early years of the 1980s, the program became a popular success and the program has been used in many industries, including:
The program has been heavily received by the federal government in several industries including manufacturing, natural gas, and natural gas, and has not been approved.
A total of 5,200 hours had been awarded with the goal of achieving a $1.5 million grant from the federal government.
The program is now

================================================================================
Python programming . He also appeared in the third season of the animated series The Simpsons . After three seasons of production , he played the character of the character of the Simpsons , and he was later credited by television producers David B. <unk> and Paul E. Lee .


 = = = Life = = =


 The Simpsons television series season was a two @-@ year @-@ long episode of the first series . The episode , which aired simultaneously on September 3 , 2002 , was the longest @-@ running episode of the series , and was the longest @-@ running episode of the series . It was viewed by 7 @.@ 7 million viewers . The episode was a 12 @-@ second @-@ year @

================================================================================
Artificial intelligence is what is a system of computer learning that has been developed, but has never been developed. It consists of some of the most important elements in the learning process, such as the ability to manipulate information. The problem of learning the most common features of the system is that the ability to detect what is happening to the machine can be measured.
A computer program is a powerful tool for learning and learning. It uses computer programming to recognize the parts of the system and the ability to interact. The program is designed to understand the parts of the system, such as what they mean to use computers to store information stored as an instrument with the ability to store information.
The system enables the computer to identify, describe, and store information in a way that will


"""

In [ ]:
#clean up for the drive deleteing unnecessary checkpoints
import os
import re

CHECKPOINT_DIR = "/content/drive/MyDrive/GPT_from_scratch/checkpoints_model2"

KEEP_MILESTONES = {
    0,
    5000,
    10000,
    15000,
    20000,
    25000
}

deleted = 0
kept = 0

for file in os.listdir(CHECKPOINT_DIR):

    if not file.endswith(".pt"):
        continue

    match = re.match(
        r"checkpoint_(\d+)\.pt",
        file
    )

    if match is None:
        continue

    step = int(
        match.group(1)
    )

    keep = (
        step in KEEP_MILESTONES
        or
        step >= 25
    )

    if keep:

        kept += 1

    else:

        os.remove(
            os.path.join(
                CHECKPOINT_DIR,
                file
            )
        )

        deleted += 1

print(
    f"Kept {kept} checkpoints"
)

print(
    f"Deleted {deleted} checkpoints"
)

##COMPARE BOTH MODELS

**tl;dr**: **Perplexity** is the gold standard metric for measuring how "surprised" a language model is by a sequence of text. Mathematically, it is the exponentiated average **cross-entropy loss**. Intuition-wise, it represents the effective "branching factor" or the number of equally likely options the model thinks it has at every step.

---

### 1. Basic Intuition: The "Branching Factor"

Imagine you are playing a game where you have to guess the next word in a sentence.

* **Low Perplexity**: The model is highly confident. It feels like you are choosing between only 2 or 3 likely words.
* **High Perplexity**: The model is "confused." It feels like you are choosing randomly from 1,000 different possibilities.

If a model has a perplexity of **10**, it means that on average, every time the model predicts the next word, it is as uncertain as if it were choosing uniformly between 10 different words.

---

### 2. The Math Behind It

To calculate perplexity, we start with the **Cross-Entropy Loss** ($L$), which measures the average negative log-likelihood of the true tokens given the model's predicted probabilities.

For a sequence of $N$ tokens $(w_1, w_2, \dots, w_N)$, the average loss is:

$$L = -\frac{1}{N} \sum_{i=1}^{N} \ln P(w_i | w_{<i})$$

**Perplexity ($PP$)** is simply $e$ raised to the power of that loss:

$$PP(w_1, \dots, w_N) = e^L = \exp\left( -\frac{1}{N} \sum_{i=1}^{N} \ln P(w_i | w_{<i}) \right)$$

By the laws of logarithms, this can also be expressed as the geometric mean of the inverse probabilities:

$$PP = \left( \prod_{i=1}^{N} \frac{1}{P(w_i | w_{<i})} \right)^{\frac{1}{N}}$$

---

### 3. Why we use $e$ (or 2)

You will see perplexity expressed as $\exp(L)$ (using natural log) or $2^L$ (if the loss was calculated using $\log_2$).

* **Bits/Token**: If you use $\log_2$, the loss represents "bits of information" required to encode the next token.
* **Natural Interpretation**: Most deep learning libraries (like PyTorch) use the natural log ($ln$) by default, so $\exp(L)$ is the standard, easy-to-compute output.

---

### 4. Key Takeaways for your GPT Project

1. **Lower is Better**: A perplexity of $1.0$ is the theoretical perfect score (the model knows exactly what comes next). Random guessing over a $50,257$ vocabulary would result in a perplexity of $50,257$.
2. **Domain Sensitivity**: A model might have "low" perplexity on Wikipedia data but "high" perplexity on your custom CS domain dataset. This tells you your model hasn't "learned" your specific domain's vocabulary or patterns yet.
3. **The "Hacker" Perspective**: Perplexity doesn't measure if the output is *true* or *helpful*; it only measures how well the model mimics the statistical distribution of the training data. A model can have great perplexity but still "hallucinate" facts perfectly fluently!

In [ ]:
"""import torch
import torch.nn.functional as F
import tiktoken
from tqdm import tqdm

@torch.inference_mode()
def compute_perplexity(model, test_data_ids, device="cuda"):
    """
    Computes Perplexity on a sequence of tokens.
    Perplexity = exp(loss)
    """
    model.eval()
    model.to(device)

    # Use the model's forward pass to calculate cross-entropy loss
    # We pass the same sequence as input and target for causal LM training
    input_ids = torch.tensor(test_data_ids, dtype=torch.long, device=device).unsqueeze(0)

    # If your forward() returns (logits, loss), use it.
    # Otherwise, compute loss from logits manually:
    logits, _ = model(input_ids)

    # Standard cross-entropy calculation
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    loss = F.cross_entropy(shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1))

    return torch.exp(loss).item()


def compare_checkpoints(checkpoint_path1, checkpoint_path2, test_text):
    tokenizer = tiktoken.get_encoding("gpt2")
    test_ids = tokenizer.encode(test_text)

    # Load Model 1
    model1 = load_production_inference_engine(checkpoint_path1)
    ppl1 = compute_perplexity(model1, test_ids)

    # Load Model 2
    model2 = load_production_inference_engine(checkpoint_path2)
    ppl2 = compute_perplexity(model2, test_ids)

    print(f"\n--- Performance Comparison ---")
    print(f"Checkpoint 1 Perplexity: {ppl1:.2f}")
    print(f"Checkpoint 2 Perplexity: {ppl2:.2f}")

    if ppl1 < ppl2:
        print(f"Model 1 is {((ppl2-ppl1)/ppl1)*100:.1f}% more confident.")
    else:
        print(f"Model 2 is {((ppl1-ppl2)/ppl2)*100:.1f}% more confident.")

# Run it:
# compare_checkpoints("path/to/v1.pt", "path/to/v2.pt", "The operating system")"""